# 05 · Roman-Empire Dataset Processing

Source: [Platonov et al., ICLR 2023](https://arxiv.org/abs/2302.11640)  
Hosted: https://github.com/yandex-research/heterophilous-graphs

**Graph description:** Word dependency graph of the Roman Empire Wikipedia article.  
Each node = one word (non-unique); edges connect consecutive words or syntactically dependent words.  

**Stats:** 
- 22,662 nodes 
- 32,927 edges (stored undirected, one copy per edge)
- 300-dim fastText features

In [1]:
import sys, os
sys.path.append('..')

import numpy as np
import pandas as pd
import torch
import urllib.request

from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

print(f'device : {"cuda" if torch.cuda.is_available() else "cpu"}')

/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device : cuda


In [2]:
input_dir  = '../data/'
output_dir = '../outputs/'
os.makedirs(input_dir,  exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

RAW_URL  = 'https://github.com/yandex-research/heterophilous-graphs/raw/main/data/roman_empire.npz'
RAW_PATH = os.path.join(input_dir, 'roman_empire_raw.npz')

## Download the Raw Dataset

In [3]:
if os.path.exists(RAW_PATH):
    print(f'Already present: {RAW_PATH}')
else:
    print(f'Downloading {RAW_URL} ...')
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)
    size_mb = os.path.getsize(RAW_PATH) / (1024 * 1024)
    print(f'Downloaded successfully — {size_mb:.2f} MB')

Downloaded successfully — 19.46 MB


## Inspect the Raw NPZ Structure

In [4]:
raw = np.load(RAW_PATH, allow_pickle=True)
print('Keys:', list(raw.keys()))
for k in raw.keys():
    v = raw[k]
    print(f'  {k:20s}  dtype={str(v.dtype):10s}  shape={v.shape}')

Keys: ['node_features', 'node_labels', 'edges', 'train_masks', 'val_masks', 'test_masks']
  node_features         dtype=float32     shape=(22662, 300)
  node_labels           dtype=int64       shape=(22662,)
  edges                 dtype=int64       shape=(32927, 2)
  train_masks           dtype=bool        shape=(10, 22662)
  val_masks             dtype=bool        shape=(10, 22662)
  test_masks            dtype=bool        shape=(10, 22662)


## Load into a DataFrame

In [6]:
node_features = raw['node_features']
node_labels = raw['node_labels']
edges = raw['edges']
train_masks = raw['train_masks']
val_masks = raw['val_masks']
test_masks = raw['test_masks']

N = node_features.shape[0]
print(f'Nodes : {N:,}')
print(f'Edges (undirec) : {edges.shape[0]:,}')
print(f'Feature dim : {node_features.shape[1]}')
print(f'Classes : {len(np.unique(node_labels))}')
print(f'Splits : {train_masks.shape[1]}')

Nodes : 22,662
Edges (undirec) : 32,927
Feature dim : 300
Classes : 18
Splits : 22662


In [8]:
df = pd.DataFrame({'node_id': np.arange(N), 'label': node_labels})
df['fasttext_embedding'] = list(node_features)

print(df.head())
print(f'\nLabel distribution:')
print(df['label'].value_counts().sort_index())

   node_id  label                                 fasttext_embedding
0        0      3  [-0.04934495, 0.09966621, 0.021878256, -0.0725...
1        1     10  [0.092196435, -0.03251098, 0.00026375614, 0.02...
2        2      6  [0.0066953376, 0.027768135, 0.035138104, -0.06...
3        3      0  [-0.0010211281, 0.046026513, 0.0060486766, 0.1...
4        4      3  [-0.051744193, 0.073963955, -0.01305688, 0.044...

Label distribution:
label
0      944
1     3163
2     3133
3     2502
4     2487
5     1359
6     1244
7     1080
8      852
9      789
10     717
11     445
12     428
13     365
14     329
15     319
16     312
17    2194
Name: count, dtype: int64


## Graph Statistics

In [12]:
import networkx as nx

# Edges are stored undirected (one copy) — build undirected graph
G = nx.Graph()
G.add_nodes_from(range(N))
G.add_edges_from(edges.tolist())

degrees = dict(G.degree())
deg_vals = list(degrees.values())

print(f'Nodes : {G.number_of_nodes():,}')
print(f'Edges (undirected) : {G.number_of_edges():,}')
print(f'Avg degree : {np.mean(deg_vals):.2f}')
print(f'Max degree : {np.max(deg_vals)}')
print(f'Is connected : {nx.is_connected(G)}')

Nodes : 22,662
Edges (undirected) : 32,927
Avg degree : 2.91
Max degree : 14
Is connected : True


## Node Feature Embeddings

In [13]:
# fastText embeddings — already in node_features, just L2-normalise to match
t = torch.from_numpy(node_features)
embeddings_ft = torch.nn.functional.normalize(t, p=2, dim=1).numpy()

norms = np.linalg.norm(embeddings_ft, axis=1)
print(f'Embedding dim : {embeddings_ft.shape[1]}')
print(f'Total nodes : {embeddings_ft.shape[0]:,}')
print(f'Zero-vectors : {(norms < 1e-8).sum()}')
print(f'Norm mean / std : {norms.mean():.4f} / {norms.std():.4f}')

Embedding dim : 300
Total nodes : 22,662
Zero-vectors : 62
Norm mean / std : 0.9973 / 0.0522


## Save Node Features CSV

In [14]:
node_features_path = input_dir + 'roman_empire_node_features.csv'
pd.DataFrame(embeddings_ft).to_csv(node_features_path, index=False, header=False)

size_mb = os.path.getsize(node_features_path) / (1024 * 1024)
print(f'node_features.csv saved — {size_mb:.2f} MB')
print(f'Shape: {embeddings_ft.shape}')

node_features.csv saved — 79.24 MB
Shape: (22662, 300)


## Generate Edge List

In [15]:
# Mirror edges: (u,v) -> both (u,v) and (v,u)
edges_both = np.concatenate([edges, edges[:, ::-1]], axis=0)
edges_both = np.unique(edges_both, axis=0)

edge_df = pd.DataFrame(edges_both, columns=['source', 'target'])
print(f'Undirected edges (stored once) : {edges.shape[0]:,}')
print(f'Bidirectional edges : {len(edge_df):,}')
print(edge_df.head())

edge_list_path = input_dir + 'roman_empire_edge_list.txt'
edge_df.to_csv(edge_list_path, sep=' ', index=False, header=False)
print(f'\nEdge list saved to {edge_list_path}')

Undirected edges (stored once) : 32,927
Bidirectional edges : 65,854
   source  target
0       0       1
1       0       2
2       1       0
3       1       2
4       2       0

Edge list saved to ../data/roman_empire_edge_list.txt


## Save Labels

In [17]:
# Labels are already 0-indexed (0–17)
labels_series = pd.Series(node_labels)
labels_path = input_dir + 'roman_empire_labels.csv'
labels_series.to_csv(labels_path, index=False, header=False)

print(f'Labels saved to {labels_path}')
print(f'Unique labels: {np.unique(node_labels)}')

unique, counts = np.unique(node_labels, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f'Class {cls}: {cnt:,} nodes')

Labels saved to ../data/roman_empire_labels.csv
Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
Class 0: 944 nodes
Class 1: 3,163 nodes
Class 2: 3,133 nodes
Class 3: 2,502 nodes
Class 4: 2,487 nodes
Class 5: 1,359 nodes
Class 6: 1,244 nodes
Class 7: 1,080 nodes
Class 8: 852 nodes
Class 9: 789 nodes
Class 10: 717 nodes
Class 11: 445 nodes
Class 12: 428 nodes
Class 13: 365 nodes
Class 14: 329 nodes
Class 15: 319 nodes
Class 16: 312 nodes
Class 17: 2,194 nodes


## Generate NPZ

In [18]:
from scripts.create_npz import create_npx

npz_path = output_dir + 'roman_empire.npz'
create_npx.create_npz_dataset(
    node_features_file=node_features_path,
    labels_file=labels_path,
    edges_file=edge_list_path,
    num_splits=10,
    output_file_name=npz_path
)

Verified Split 0 - Train: 13597, Val: 4532, Test: 4533
no of nodes 22662
no of features per node 300
no of node labels, should matach no of nodes 22662
no of edges 65854


In [19]:
size_mb = os.path.getsize(npz_path) / (1024 * 1024)
print(f'NPZ file size: {size_mb:.2f} MB')

NPZ file size: 27.76 MB


## Validation

In [20]:
data = np.load(npz_path)
print('Keys in NPZ:', list(data.keys()))
for key in data.keys():
    print(f'  {key} shape: {data[key].shape}')

labels_out = data['node_labels']
unique, counts = np.unique(labels_out, return_counts=True)
print('Label distribution in NPZ:')
print(dict(zip(unique.tolist(), counts.tolist())))

Keys in NPZ: ['node_features', 'node_labels', 'edges', 'train_masks', 'val_masks', 'test_masks']
  node_features shape: (22662, 300)
  node_labels shape: (22662,)
  edges shape: (65854, 2)
  train_masks shape: (10, 22662)
  val_masks shape: (10, 22662)
  test_masks shape: (10, 22662)
Label distribution in NPZ:
{0: 944, 1: 3163, 2: 3133, 3: 2502, 4: 2487, 5: 1359, 6: 1244, 7: 1080, 8: 852, 9: 789, 10: 717, 11: 445, 12: 428, 13: 365, 14: 329, 15: 319, 16: 312, 17: 2194}


## Similarity Sanity Check

In [ ]:
def test_word_similarity(embeddings, node_ids, top_k=5):
    query_idx = node_ids[0]
    query_vec = embeddings[query_idx].reshape(1, -1)

    scores  = cosine_similarity(query_vec, embeddings).flatten()
    indices = np.argsort(scores)[-(top_k + 1):][::-1]

    print(f'Query node index : {query_idx}')
    print(f'Query label : {node_labels[query_idx]}')
    print(f'\nTop {top_k} most similar nodes:')
    for i in indices:
        if i == query_idx:
            continue
        print(f'  [sim={scores[i]:.4f}]  node_id={i}  label={node_labels[i]}')

test_word_similarity(embeddings_ft, [100, 500, 1000])

Query node index : 100
Query label : 14

Top 5 most similar nodes:
  [sim=0.5405]  node_id=16408  label=5
  [sim=0.4931]  node_id=16872  label=4
  [sim=0.4931]  node_id=8812  label=4
  [sim=0.4759]  node_id=16106  label=5
  [sim=0.4759]  node_id=16651  label=1
